# HumAID — Zero-shot Classification (Filtered Labels, Batch API, Sharding, Stand Alone)

- **Filtered labels (per event):** prompts + JSON schema only list labels that appear in that event’s ground truth → reduces out-of-scope (OOS) predictions.
- **Batch API flow:** build `requests.jsonl` → upload → create batch → poll → download `outputs.jsonl` (and `errors.jsonl` if any).
- **Patch pass:** after batch completes, any missing/blank predictions are re-classified synchronously so `predictions.csv` has one row per input.
- **Stratified sharding (optional):** split large events into *k* shards **preserving class ratios**; use the **same** event-level labels + rules for all shards; merge predictions back in original order.
- **Reporting:** confusion matrices (counts + row-normalized), per-class F1/error, mistakes CSV, and a sortable `results/index.html`.  
  - **Scope** = label universe used for metrics (default `truth`).  
  - **OOS preds** = predictions not in the truth set (QA signal).

## Key settings
- `MODEL` (e.g., `gpt-4o`), `RULES` (e.g., `RULES_1`), `TAG`
- `DRYRUN_N`, `POLL_SECS`
- Token budgeting: `BATCH_TOKEN_LIMIT`, `SAFETY_MARGIN`, `MAX_OUTPUT_TOKENS`
- `.env` with `OPENAI_API_KEY_1` (and optionally a second key)

# 0) Setup

In [1]:
# test_gpt_unified.py - Unified classifier supporting both new (GPT-5) and old (GPT-4) APIs
# Works with both /responses endpoint (GPT-5) and /chat/completions endpoint (GPT-4-0613)
# Usage: python test_gpt_unified.py --tsv Dataset/HumAID/event_x/event_x_test.tsv --model gpt-4-0613

import os, json, re, time, argparse, uuid, requests
import pandas as pd
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv; load_dotenv()

# Try to import RULES_1, provide fallback if not available
try:
    from rules import RULES_1
except ImportError:
    RULES_1 = ""

SYSTEM_PROMPT = (
  "You are a precise tweet classifier for humanitarian-response content.\n"
  "Choose exactly one label from the allowed labels.\n"
  "If unrelated to humanitarian contexts, choose 'not_humanitarian'.\n"
  "Never invent labels not listed in the allowed labels.\n"
  "Output JSON matching the provided schema; no extra fields."
)

HUMAID_LABELS = [
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "infrastructure_and_utility_damage",
    "injured_or_dead_people",
    "missing_or_found_people",
    "requests_or_urgent_needs",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
    "other_relevant_information",
    "not_humanitarian",
]

# Models that support the newer /responses endpoint with structured outputs
RESPONSES_MODELS = ["gpt-5", "gpt-5-mini", "gpt-4o-2024-08-06", "gpt-4o-mini-2024-07-18"]

def filter_labels_in_scope(labels: list[str]) -> list[str]:
    """Preserve incoming order, keep only known HumAID labels, and dedupe."""
    seen = set()
    out = []
    for l in labels:
        if l in HUMAID_LABELS and l not in seen:
            seen.add(l)
            out.append(l)
    return out
    
def parse_rules_blocks(rules_text: str) -> dict[str, str]:
    """
    Expects blocks in the format:
    - label_name
      Definition: ...
      Include: ...
      Exclude: ...
    """
    blocks = {}
    # Split on lines that start with "- <label>"
    parts = re.split(r'\n(?=-\s+[a-z0-9_]+)', "\n"+rules_text.strip(), flags=re.I)
    for p in parts:
        m = re.match(r'-\s+([a-z0-9_]+)\s*\n(.+)$', p.strip(), flags=re.I|re.S)
        if m:
            label = m.group(1).strip()
            body  = m.group(2).rstrip()
            blocks[label] = f"- {label}\n{body}\n"
    return blocks

def slice_rules_for_labels(rules_text: str, labels: list[str]) -> str:
    blocks = parse_rules_blocks(rules_text)
    kept   = [blocks[l] for l in labels if l in blocks]
    return ("\n".join(kept)).strip()

def _infer_event_split(tsv_path: str) -> tuple[str, str]:
    p = Path(tsv_path)
    event = p.parent.name                       # e.g., kerala_floods_2018
    m = re.search(r'_(train|dev|test)\.tsv$', p.name, flags=re.I)
    split = m.group(1).lower() if m else "unknown"
    return event, split

def plan_dirs_like_package(tsv_path: str, out_root: str, model: str, tag: str):
    event, split = _infer_event_split(tsv_path)
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    run_dir = Path(out_root) / event / split / model / f"{stamp}-{tag}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return {
        "dir": run_dir,
        "predictions_csv": run_dir / "predictions.csv",
        "summary_json":    run_dir / "summary.json",
        "meta_json":       run_dir / "meta.json",
    }

# ---------- Config ----------
OPENAI_BASE = "https://api.openai.com/v1"
API_KEY = os.getenv("OPENAI_API_KEY_1") or os.getenv("OPENAI_API_KEY")
assert API_KEY, "Set OPENAI_API_KEY_1 or OPENAI_API_KEY in your env."
HEAD = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}

# Retry configuration
MAX_RETRY_ATTEMPTS = 5
RETRY_DELAY = 1.0

# ---------- IO ----------
def load_tsv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    # Normalize required columns
    if "tweet_id" not in df.columns or "tweet_text" not in df.columns:
        raise ValueError("TSV must contain 'tweet_id' and 'tweet_text'.")
    # Optional ground truth
    if "class_label" not in df.columns:
        df["class_label"] = ""
    df["tweet_id"] = df["tweet_id"].astype(str)
    df["tweet_text"] = df["tweet_text"].astype(str)
    df["class_label"] = df["class_label"].astype(str)
    return df

def event_labels(df: pd.DataFrame, supplied: list[str] | None = None) -> list[str]:
    if supplied:
        return filter_labels_in_scope(list(supplied))

    # Collect unique labels from the dataset in the order of first appearance
    seen = set()
    ordered_truth = []
    for lab in df["class_label"].astype(str):
        lab = lab.strip()
        if lab and lab.lower() not in {"nan", "none"} and lab not in seen:
            seen.add(lab)
            ordered_truth.append(lab)

    # If no ground truth labels (inference-only), fall back to full HumAID universe
    if not ordered_truth:
        return HUMAID_LABELS.copy()

    return filter_labels_in_scope(ordered_truth)

# ---------- Schema / parsing ----------
def make_schema(labels: list[str], keep_conf: bool=False) -> dict:
    # Schema for newer models that support structured outputs
    props = {"label": {"type": "string", "enum": labels}}
    if keep_conf:
        props["confidence"] = {"type": "number", "minimum": 0, "maximum": 1}
    return {
        "type": "object",
        "properties": props,
        "required": ["label"],
        "additionalProperties": False,
    }

def normalize_label(text: str, labels: list[str]) -> str | None:
    if not text:
        return None
    t = text.strip().strip('"\'')

    # Exact (case-insensitive)
    for L in labels:
        if t.lower() == L.lower():
            return L

    # Try JSON fragment
    m = re.search(r'"label"\s*:\s*"([^"]+)"', t)
    if m:
        cand = m.group(1).strip()
        for L in labels:
            if cand.lower() == L.lower():
                return L

    # Try loose "label: X"
    m = re.search(r'\blabel\s*:\s*([A-Za-z0-9\-\_]+)', t)
    if m:
        cand = m.group(1).strip()
        for L in labels:
            if cand.lower() == L.lower():
                return L

    # Fuzzy contains (last resort)
    for L in labels:
        if L.lower() in t.lower():
            return L
    return None

def extract_from_responses(resp_json: dict, labels: list[str]) -> tuple[str | None, dict]:
    """Extract label from /responses endpoint response"""
    # Preferred: output_parsed (Structured Outputs on Responses API)
    op = resp_json.get("output_parsed")
    if isinstance(op, list) and op:
        obj = op[0]
        if isinstance(obj, dict) and "label" in obj:
            return str(obj["label"]).strip(), obj

    # Fallback: the rendered message text
    text = ""
    out = resp_json.get("output", [])
    try:
        for item in out:
            if item.get("type") == "message":
                parts = item.get("content", [])
                if parts and isinstance(parts, list):
                    for p in parts:
                        if isinstance(p, dict) and "text" in p:
                            text = p["text"]
                            break
                if text:
                    break
        if not text and out:
            parts = out[0].get("content", [])
            if parts and isinstance(parts, list):
                for p in parts:
                    if isinstance(p, dict) and "text" in p:
                        text = p["text"]
                        break
    except Exception:
        text = ""

    # Try to parse JSON
    if text:
        try:
            obj = json.loads(text)
            if isinstance(obj, dict) and "label" in obj:
                return str(obj["label"]).strip(), obj
        except Exception:
            pass

    lab = normalize_label(text, labels)
    return lab, ({"label": lab} if lab else {})

def extract_from_chat_completion(resp_json: dict, labels: list[str]) -> tuple[str | None, dict]:
    """Extract label from /chat/completions endpoint response"""
    try:
        # Get the assistant's message content
        message = resp_json.get("choices", [{}])[0].get("message", {})
        content = message.get("content", "")
        
        # First try to parse as JSON
        try:
            obj = json.loads(content)
            if isinstance(obj, dict) and "label" in obj:
                label = str(obj["label"]).strip()
                # Validate the label
                if label in labels:
                    return label, obj
                # Try case-insensitive match
                for L in labels:
                    if label.lower() == L.lower():
                        return L, {"label": L}
        except json.JSONDecodeError:
            pass
        
        # Fall back to text parsing
        lab = normalize_label(content, labels)
        return lab, ({"label": lab} if lab else {})
        
    except Exception as e:
        print(f"  Error extracting from chat completion: {e}")
        return None, {"error": str(e)}

def supports_responses_api(model: str) -> bool:
    """Check if model supports the newer /responses endpoint"""
    # Check if model starts with any known responses-capable model prefix
    for rm in RESPONSES_MODELS:
        if model.startswith(rm):
            return True
    return False

# ---------- API with RETRY (Unified for both endpoints) ----------
def classify_row_unified(model: str, text: str, labels: list[str], rules: str, 
                         max_output_tokens: int = 500,
                         timeout: int = 60, poll_retries: int = 15, poll_delay: float = 2.0) -> tuple[str | None, dict]:
    """
    Unified classification that works with both /responses (GPT-5) and /chat/completions (GPT-4) endpoints.
    Returns (label, parsed_dict)
    """
    
    use_responses = supports_responses_api(model)
    
    for attempt in range(MAX_RETRY_ATTEMPTS):
        try:
            # Build the user prompt (same for both endpoints)
            if attempt == 0:
                user_prompt = (
                    "Classify this tweet into exactly ONE of the allowed labels.\n"
                    f"Allowed labels: {', '.join(labels)}\n\n"
                    "Output your response as a JSON object with a 'label' field.\n"
                    "Example: {\"label\": \"not_humanitarian\"}\n\n"
                    "Rules:\n"
                    f"{rules}\n\n" 
                    f'Tweet: """{text.strip()}"""'  
                )
            else:
                # Stronger emphasis on retry
                user_prompt = (
                    "IMPORTANT: You MUST choose EXACTLY ONE label from this list:\n"
                    f"[{', '.join(labels)}]\n\n"
                    "Do NOT make up labels. ONLY use the labels provided above.\n"
                    "Output as JSON: {\"label\": \"your_chosen_label\"}\n\n"
                    "Rules:\n"
                    f"{rules}\n\n" 
                    f'Tweet: """{text.strip()}"""'  
                )
            
            if use_responses:
                # Use newer /responses endpoint with structured outputs
                schema = make_schema(labels, keep_conf=False)
                body = {
                    "model": model,
                    "input": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_prompt},
                    ],
                    "text": {
                        "format": {
                            "type": "json_schema",
                            "name": "tweet_label",
                            "schema": schema,
                            "strict": True
                        }
                    },
                    "max_output_tokens": max_output_tokens
                }

                r = requests.post(f"{OPENAI_BASE}/responses", headers=HEAD, json=body, timeout=timeout)
                
                if r.status_code != 200:
                    if attempt < MAX_RETRY_ATTEMPTS - 1:
                        print(f"  ⚠ HTTP {r.status_code} on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}, retrying...")
                        time.sleep(RETRY_DELAY)
                        continue
                    return None, {"error": f"HTTP {r.status_code}: {r.text[:200]}", "attempts": attempt + 1}

                j = r.json()
                status = j.get("status")
                
                if status == "incomplete":
                    rid = j.get("id")
                    for _ in range(poll_retries):
                        time.sleep(poll_delay)
                        rr = requests.get(f"{OPENAI_BASE}/responses/{rid}", 
                                        headers={"Authorization": f"Bearer {API_KEY}"}, 
                                        timeout=timeout)
                        if rr.status_code != 200:
                            continue
                        j = rr.json()
                        if j.get("status") == "completed":
                            break
                        if j.get("status") == "failed":
                            if attempt < MAX_RETRY_ATTEMPTS - 1:
                                print(f"  ⚠ Response failed on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}, retrying...")
                                time.sleep(RETRY_DELAY)
                                break
                            return None, {"error": f"Response failed: {j.get('error')}", "attempts": attempt + 1}

                label, parsed = extract_from_responses(j, labels)
                
            else:
                # Use standard /chat/completions endpoint for older models
                body = {
                    "model": model,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": user_prompt}
                    ],
                    "max_tokens": max_output_tokens,
                    "temperature": 0.0,  # Make it more deterministic
                    "response_format": {"type": "json_object"} if model.startswith("gpt-4-1106") else None
                }
                
                # Remove None values
                body = {k: v for k, v in body.items() if v is not None}
                
                r = requests.post(f"{OPENAI_BASE}/chat/completions", headers=HEAD, json=body, timeout=timeout)
                
                if r.status_code != 200:
                    if attempt < MAX_RETRY_ATTEMPTS - 1:
                        print(f"  ⚠ HTTP {r.status_code} on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}, retrying...")
                        time.sleep(RETRY_DELAY)
                        continue
                    error_msg = r.text[:500] if r.text else f"HTTP {r.status_code}"
                    return None, {"error": error_msg, "attempts": attempt + 1}

                j = r.json()
                label, parsed = extract_from_chat_completion(j, labels)
            
            # Validate the label
            if label and label in labels:
                if attempt > 0:
                    print(f"  ✓ Got valid label '{label}' on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}")
                return label, parsed
            
            # Invalid label, retry if we have attempts left
            if attempt < MAX_RETRY_ATTEMPTS - 1:
                print(f"  ⚠ Invalid label '{label}' on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}, retrying...")
                time.sleep(RETRY_DELAY)
            else:
                print(f"  ✗ Failed to get valid label after {MAX_RETRY_ATTEMPTS} attempts (got: '{label}')")
                # Use fallback to not_humanitarian if available
                if "not_humanitarian" in labels:
                    print(f"    → Using fallback: 'not_humanitarian'")
                    return "not_humanitarian", {"fallback": True, "original": label, "attempts": MAX_RETRY_ATTEMPTS}
                return None, {"error": f"Invalid label after {MAX_RETRY_ATTEMPTS} attempts", "last_label": label, "attempts": MAX_RETRY_ATTEMPTS}
                
        except Exception as e:
            if attempt < MAX_RETRY_ATTEMPTS - 1:
                print(f"  ⚠ Exception on attempt {attempt + 1}/{MAX_RETRY_ATTEMPTS}: {e}, retrying...")
                time.sleep(RETRY_DELAY)
            else:
                return None, {"error": f"Exception: {str(e)}", "attempts": attempt + 1}
    
    return None, {"error": f"Max attempts ({MAX_RETRY_ATTEMPTS}) reached"}

# ---------- Metrics ----------
def compute_metrics(df: pd.DataFrame, col_truth="class_label", col_pred="predicted_label") -> dict:
    """
    Compute metrics properly, handling NaN values.
    """
    try:
        from sklearn.metrics import f1_score, classification_report
        
        # Clean data
        df_clean = df.copy()
        for col in [col_truth, col_pred]:
            df_clean[col] = df_clean[col].astype(str).str.strip()
            df_clean[col] = df_clean[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
        df_clean = df_clean.dropna(subset=[col_truth, col_pred])
        
        if df_clean.empty:
            return {"macro_f1": 0.0, "weighted_f1": 0.0, "num_evaluated": 0, "num_invalid": len(df)}
            
        y_true = df_clean[col_truth].tolist()
        y_pred = df_clean[col_pred].tolist()
        truth_labels = sorted(set(y_true))
        
        macro = f1_score(y_true, y_pred, labels=truth_labels, average="macro", zero_division=0)
        weighted = f1_score(y_true, y_pred, labels=truth_labels, average="weighted", zero_division=0)
        rep = classification_report(y_true, y_pred, labels=truth_labels, output_dict=True, zero_division=0)
        
        return {
            "macro_f1": macro, 
            "weighted_f1": weighted, 
            "per_class": rep,
            "num_evaluated": len(df_clean),
            "num_invalid": len(df) - len(df_clean)
        }
    except Exception as e:
        return {"error": str(e)}

# ---------- Main single-event runner ----------
def run_single_event(tsv_path: str,
                     model: str = "gpt-4-0613",
                     rules: str = "",
                     out_root: str = "runs",
                     tag: str = "unified-classifier",
                     max_rows: int | None = None):
    """
    Run single event classification with automatic retry on invalid labels.
    Works with both new models (GPT-5) and older models (GPT-4-0613).
    """
    df = load_tsv(tsv_path)
    if max_rows:
        df = df.head(max_rows).copy()

    plan = plan_dirs_like_package(tsv_path, out_root=out_root, model=model, tag=tag)

    labels = event_labels(df)
    print(f"Event: {Path(tsv_path).parent.name}")
    print(f"Model: {model}")
    print(f"API Endpoint: {'responses' if supports_responses_api(model) else 'chat/completions'}")
    print(f"Labels in scope: {labels}")

    # Default rules if none provided
    if not rules:
        rules = RULES_1 if RULES_1 else ""

    # Slice rules to only the labels in this event schema
    rules_scoped = slice_rules_for_labels(rules, labels) if rules else ""
    
    print(f"Processing {len(df)} rows with up to {MAX_RETRY_ATTEMPTS} attempts per row...")

    # Track statistics
    stats = {
        "total_rows": len(df),
        "successful_first_try": 0,
        "successful_with_retry": 0,
        "used_fallback": 0,
        "failed_completely": 0,
    }

    # single-label fast path
    if len(labels) == 1:
        only = labels[0]
        out = df[["tweet_id","tweet_text","class_label"]].copy()
        out["predicted_label"] = only
        out["confidence"] = 1.0
        out.to_csv(plan["predictions_csv"], index=False)
        metrics = compute_metrics(out)
        metrics["mode"] = "single-label-fast-path"
        with open(plan["summary_json"], "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2)
        with open(plan["meta_json"], "w", encoding="utf-8") as f:
            json.dump({"model": model, "tag": tag, "tsv": str(tsv_path)}, f, indent=2)
        print(f"Single label detected, using fast path: {only}")
        return out, metrics

    # multi-label path with retry logic
    rows = []
    for i, r in df.iterrows():
        lab, parsed = classify_row_unified(
            model, r["tweet_text"], labels, rules_scoped,
            max_output_tokens=500, timeout=60,
            poll_retries=15, poll_delay=2.0
        )
        
        # Track statistics
        attempts = parsed.get("attempts", 1) if isinstance(parsed, dict) else 1
        if lab and lab in labels:
            if parsed.get("fallback"):
                stats["used_fallback"] += 1
            elif attempts == 1:
                stats["successful_first_try"] += 1
            else:
                stats["successful_with_retry"] += 1
        else:
            stats["failed_completely"] += 1
            lab = pd.NA
        
        rows.append({
            "tweet_id": r["tweet_id"],
            "tweet_text": r["tweet_text"],
            "class_label": r["class_label"],
            "predicted_label": lab if pd.notna(lab) else pd.NA,
            "confidence": parsed.get("confidence") if isinstance(parsed, dict) else None,
        })
        
        if (i+1) % 20 == 0:
            success = stats["successful_first_try"] + stats["successful_with_retry"] + stats["used_fallback"]
            rate = success / (i+1)
            print(f"Processed {i+1}/{len(df)} rows... (success rate: {rate:.1%})")

    out = pd.DataFrame(rows)
    out.to_csv(plan["predictions_csv"], index=False)

    metrics = compute_metrics(out)
    metrics["classification_stats"] = stats
    metrics["api_endpoint"] = "responses" if supports_responses_api(model) else "chat/completions"
    
    with open(plan["summary_json"], "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    
    with open(plan["meta_json"], "w", encoding="utf-8") as f:
        json.dump({
            "model": model, 
            "tag": tag, 
            "tsv": str(tsv_path),
            "max_retry_attempts": MAX_RETRY_ATTEMPTS,
            "retry_delay": RETRY_DELAY,
            "api_endpoint": metrics["api_endpoint"]
        }, f, indent=2)

    print("\n" + "="*50)
    print("CLASSIFICATION COMPLETE")
    print("="*50)
    print(f"Total rows: {stats['total_rows']}")
    print(f"Success on first try: {stats['successful_first_try']} ({stats['successful_first_try']/stats['total_rows']:.1%})")
    print(f"Success with retry: {stats['successful_with_retry']} ({stats['successful_with_retry']/stats['total_rows']:.1%})")
    print(f"Used fallback: {stats['used_fallback']} ({stats['used_fallback']/stats['total_rows']:.1%})")
    print(f"Failed completely: {stats['failed_completely']} ({stats['failed_completely']/stats['total_rows']:.1%})")
    
    if metrics.get("macro_f1") is not None:
        print(f"\nMacro F1: {metrics['macro_f1']:.4f}")
        if metrics.get("weighted_f1") is not None:
            print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
        if metrics.get("num_invalid"):
            print(f"Invalid predictions excluded: {metrics['num_invalid']}")
    
    print(f"\nResults saved to: {plan['predictions_csv']}")
    return out, metrics

# # ---------- Script entry ----------
# if __name__ == "__main__":
#     p = argparse.ArgumentParser()
#     p.add_argument("--tsv", required=True, help="Path to a single event TSV (tweet_id, tweet_text, optional class_label).")
#     p.add_argument("--model", default="gpt-4-0613", help="Model name, e.g., gpt-4-0613, gpt-5-mini, gpt-5.")
#     p.add_argument("--rules", default="", help="Rules text to include in the prompt (or 'RULES_1').")
#     p.add_argument("--out_root", default="runs", help="Root directory for outputs.")
#     p.add_argument("--tag", default="unified-classifier", help="Tag for the run folder.")
#     p.add_argument("--max_rows", type=int, default=None, help="Limit rows for a quick test.")
#     args = p.parse_args()
    
#     # Load rules if specified
#     rules_text = args.rules
#     if args.rules == "RULES_1":
#         rules_text = RULES_1
    
#     run_single_event(
#         args.tsv, 
#         model=args.model, 
#         rules=rules_text, 
#         out_root=args.out_root,
#         tag=args.tag,
#         max_rows=args.max_rows
#     )

In [2]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-4-0613",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-4-0613-RULES1-filtered",
    max_rows=None
)

Event: kaikoura_earthquake_2016
Model: gpt-4-0613
API Endpoint: chat/completions
Labels in scope: ['infrastructure_and_utility_damage', 'sympathy_and_support', 'not_humanitarian', 'caution_and_advice', 'other_relevant_information', 'injured_or_dead_people', 'rescue_volunteering_or_donation_effort', 'displaced_people_and_evacuations', 'requests_or_urgent_needs']
Processing 435 rows with up to 5 attempts per row...
Processed 20/435 rows... (success rate: 100.0%)
Processed 40/435 rows... (success rate: 100.0%)
  ⚠ HTTP 503 on attempt 1/5, retrying...
  ✓ Got valid label 'caution_and_advice' on attempt 2/5
Processed 60/435 rows... (success rate: 100.0%)
Processed 80/435 rows... (success rate: 100.0%)
Processed 100/435 rows... (success rate: 100.0%)
Processed 120/435 rows... (success rate: 100.0%)
Processed 140/435 rows... (success rate: 100.0%)
Processed 160/435 rows... (success rate: 100.0%)
Processed 180/435 rows... (success rate: 100.0%)
Processed 200/435 rows... (success rate: 100.0%)


In [2]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-4-0314",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-4-0314-RULES1-filtered",
    max_rows=None
)

Event: kaikoura_earthquake_2016
Model: gpt-4-0314
API Endpoint: chat/completions
Labels in scope: ['infrastructure_and_utility_damage', 'sympathy_and_support', 'not_humanitarian', 'caution_and_advice', 'other_relevant_information', 'injured_or_dead_people', 'rescue_volunteering_or_donation_effort', 'displaced_people_and_evacuations', 'requests_or_urgent_needs']
Processing 435 rows with up to 5 attempts per row...
  ⚠ HTTP 404 on attempt 1/5, retrying...
  ⚠ HTTP 404 on attempt 2/5, retrying...
  ⚠ HTTP 404 on attempt 3/5, retrying...
  ⚠ HTTP 404 on attempt 4/5, retrying...


KeyboardInterrupt: 

In [ ]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-5",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-5-RULES1-filtered",
    max_rows=None
)

In [ ]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-5",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-5-RULES1-filtered",
    max_rows=None
)

In [ ]:
from rules import RULES_1

out, metrics = run_single_event(
    "Dataset/HumAID/kaikoura_earthquake_2016/kaikoura_earthquake_2016_test.tsv",
    model="gpt-5",
    rules=RULES_1,
    out_root="runs",
    tag="modeR-gpt-5-RULES1-filtered",
    max_rows=None
)